<a href="https://colab.research.google.com/github/guilhermek32/ListaVisao-2026.1/blob/Lista-03/Lista_03_Q1_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Dupla: Antonio Guilherme da Silva e Jean Patrick Martins Almeida
---



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
from pathlib import Path
from scipy import ndimage
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

# Configuração para exibir gráficos no Jupyter Notebook
%matplotlib inline


# Questão 1:

Escolha uma das metodologias implementadas na segunda lista para gerar correspondências entre um par de imagens e implemente um algoritmo que calcule homografias entre pares de imagens a partir de correspondências.

Aplique este algoritmo para gerar panoramas a partir de trios de imagens, com sobreposição 2 a 2, alinhando as imagens no plano da primeira imagem. Repita o mesmo alinhando no plano da segunda imagem e da terceira imagem.

Aqui é necessário compor transformações de homografia em alguns casos, ou calcular inversas. Gere panoramas para 5 trios de imagens distintos. Não use a API de alto nível `Stitcher`. Use `warpPerspective`.


In [ ]:
from pathlib import Path



RNG = np.random.default_rng(42)


def load_image(path):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(f"Não foi possível ler: {path}")
    return image


def match_features(img1_gray, img2_gray, method="SIFT", max_matches=2000):
    """Correspondencias baseadas na função de matching da Lista 02."""
    if method == "SIFT":
        descriptor = cv2.SIFT_create()
        norm_type = cv2.NORM_L2
    elif method == "ORB":
        descriptor = cv2.ORB_create(nfeatures=4000)
        norm_type = cv2.NORM_HAMMING
    elif method == "BRISK":
        descriptor = cv2.BRISK_create()
        norm_type = cv2.NORM_HAMMING
    else:
        raise ValueError("metodo deve ser SIFT, ORB ou BRISK")

    kp1, des1 = descriptor.detectAndCompute(img1_gray, None)
    kp2, des2 = descriptor.detectAndCompute(img2_gray, None)
    if des1 is None or des2 is None:
        return [], kp1, kp2

    bf = cv2.BFMatcher(norm_type, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    return matches[:max_matches], kp1, kp2


def normalize_points(points):
    points = np.asarray(points, dtype=np.float64)
    mean = points.mean(axis=0)
    centered = points - mean
    mean_dist = np.mean(np.linalg.norm(centered, axis=1))
    scale = np.sqrt(2.0) / mean_dist if mean_dist > 1e-12 else 1.0
    transform = np.array(
        [[scale, 0.0, -scale * mean[0]], [0.0, scale, -scale * mean[1]], [0.0, 0.0, 1.0]]
    )
    homogeneous = np.column_stack([points, np.ones(len(points))])
    normalized = (transform @ homogeneous.T).T[:, :2]
    return normalized, transform


def compute_homography_dlt(src_points, dst_points):
    """DLT normalizado: estima H tal que dst ~= H * src."""
    src_points = np.asarray(src_points, dtype=np.float64)
    dst_points = np.asarray(dst_points, dtype=np.float64)
    if len(src_points) < 4:
        raise ValueError("Homografia precisa de pelo menos 4 correspondências")

    src_norm, t_src = normalize_points(src_points)
    dst_norm, t_dst = normalize_points(dst_points)

    a = []
    for (x, y), (u, v) in zip(src_norm, dst_norm):
        a.append([-x, -y, -1.0, 0.0, 0.0, 0.0, u * x, u * y, u])
        a.append([0.0, 0.0, 0.0, -x, -y, -1.0, v * x, v * y, v])
    a = np.asarray(a, dtype=np.float64)

    _, _, vt = np.linalg.svd(a)
    h_norm = vt[-1].reshape(3, 3)
    h = np.linalg.inv(t_dst) @ h_norm @ t_src
    return h / h[2, 2]


def project_points(points, h):
    points_h = np.column_stack([points, np.ones(len(points))])
    projected = (h @ points_h.T).T
    projected = projected[:, :2] / projected[:, 2:3]
    return projected


def compute_translation_homography(src_points, dst_points):
    """Fallback robusto: translação mediana estimada das correspondências."""
    deltas = np.asarray(dst_points, dtype=np.float64) - np.asarray(src_points, dtype=np.float64)
    # Em panoramas horizontais consecutivos, a imagem seguinte costuma deslocar
    # os pontos para a esquerda. O filtro reduz matches repetidos ruins em telhados.
    likely = deltas[(deltas[:, 0] < -100.0) & (np.abs(deltas[:, 1]) < 1000.0)]
    if len(likely) >= 20:
        deltas = likely
    dx, dy = np.median(deltas, axis=0)
    return np.array([[1.0, 0.0, dx], [0.0, 1.0, dy], [0.0, 0.0, 1.0]], dtype=np.float64)


def ransac_homography(src_points, dst_points, iterations=500, threshold=4.0):
    src_points = np.asarray(src_points, dtype=np.float64)
    dst_points = np.asarray(dst_points, dtype=np.float64)
    if len(src_points) < 4:
        raise ValueError("Poucas correspondencias para RANSAC")

    best_inliers = None
    best_count = -1
    indices = np.arange(len(src_points))

    for _ in range(iterations):
        sample = RNG.choice(indices, size=4, replace=False)
        try:
            h = compute_homography_dlt(src_points[sample], dst_points[sample])
        except np.linalg.LinAlgError:
            continue
        projected = project_points(src_points, h)
        errors = np.linalg.norm(projected - dst_points, axis=1)
        inliers = errors < threshold
        count = int(inliers.sum())
        if count > best_count:
            best_count = count
            best_inliers = inliers

    if best_inliers is None or best_count < 4:
        raise RuntimeError("RANSAC não encontrou homografia válida")

    h = compute_homography_dlt(src_points[best_inliers], dst_points[best_inliers])
    return h, best_inliers


def estimate_pair_homography(
    img_a,
    img_b,
    method,
    output_debug_prefix=None,
    ransac_iterations=500,
    min_inliers=100,
    min_inlier_ratio=0.2,
):
    gray_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)
    matches, kp_a, kp_b = match_features(gray_a, gray_b, method=method)
    if len(matches) < 4:
        raise RuntimeError(f"Poucos matches: {len(matches)}")

    src = np.float64([kp_a[m.queryIdx].pt for m in matches])
    dst = np.float64([kp_b[m.trainIdx].pt for m in matches])
    h, inliers = ransac_homography(src, dst, iterations=ransac_iterations)
    used_fallback = False
    inlier_count = int(inliers.sum())
    inlier_ratio = inlier_count / len(matches)
    if inlier_count < min_inliers or inlier_ratio < min_inlier_ratio:
        h = compute_translation_homography(src, dst)
        projected = project_points(src, h)
        errors = np.linalg.norm(projected - dst, axis=1)
        inliers = errors < 25.0
        used_fallback = True

    if output_debug_prefix:
        inlier_matches = [m for m, ok in zip(matches, inliers) if ok]
        debug = cv2.drawMatches(
            img_a,
            kp_a,
            img_b,
            kp_b,
            inlier_matches[:80],
            None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
        )
        cv2.imwrite(f"{output_debug_prefix}_matches.png", debug)

    return h, len(matches), int(inliers.sum()), used_fallback


def estimate_pair_translation(img_a, img_b, method):
    gray_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)
    matches, kp_a, kp_b = match_features(gray_a, gray_b, method=method)
    if len(matches) < 4:
        raise RuntimeError(f"Poucos matches: {len(matches)}")
    src = np.float64([kp_a[m.queryIdx].pt for m in matches])
    dst = np.float64([kp_b[m.trainIdx].pt for m in matches])
    return compute_translation_homography(src, dst)


def transformed_corners(image, h):
    height, width = image.shape[:2]
    corners = np.float64([[0, 0], [width, 0], [width, height], [0, height]])
    return project_points(corners, h)


def make_panorama(images, transforms_to_reference, max_side=20000, max_area=120_000_000):
    all_corners = np.vstack(
        [transformed_corners(image, h) for image, h in zip(images, transforms_to_reference)]
    )
    min_xy = np.floor(all_corners.min(axis=0)).astype(int)
    max_xy = np.ceil(all_corners.max(axis=0)).astype(int)

    translation = np.array(
        [[1.0, 0.0, -min_xy[0]], [0.0, 1.0, -min_xy[1]], [0.0, 0.0, 1.0]],
        dtype=np.float64,
    )
    canvas_size = (int(max_xy[0] - min_xy[0]), int(max_xy[1] - min_xy[1]))
    canvas_area = canvas_size[0] * canvas_size[1]
    if canvas_size[0] > max_side or canvas_size[1] > max_side or canvas_area > max_area:
        raise RuntimeError(
            "canvas muito grande "
            f"({canvas_size[0]}x{canvas_size[1]}). Homografia provavelmente instável."
        )

    panorama = np.zeros((canvas_size[1], canvas_size[0], 3), dtype=np.uint8)
    occupied = np.zeros((canvas_size[1], canvas_size[0]), dtype=bool)

    for image, h in zip(images, transforms_to_reference):
        h_canvas = translation @ h
        warped = cv2.warpPerspective(image, h_canvas, canvas_size)
        mask = cv2.warpPerspective(
            np.ones(image.shape[:2], dtype=np.uint8) * 255, h_canvas, canvas_size
        )
        mask = mask > 0

        new_pixels = mask & ~occupied
        overlap = mask & occupied
        panorama[new_pixels] = warped[new_pixels]
        if np.any(overlap):
            panorama[overlap] = cv2.addWeighted(panorama[overlap], 0.5, warped[overlap], 0.5, 0)
        occupied |= mask

    return panorama


def process_trio(paths, method, output_dir, ransac_iterations, min_inliers, min_inlier_ratio):
    output_dir.mkdir(parents=True, exist_ok=True)
    images = [load_image(path) for path in paths]
    stem = paths[0].stem.rsplit("_", 1)[0]

    h12, matches12, inliers12, fallback12 = estimate_pair_homography(
        images[0], images[1], method, output_dir / f"{stem}_01_02", ransac_iterations, min_inliers, min_inlier_ratio
    )
    h23, matches23, inliers23, fallback23 = estimate_pair_homography(
        images[1], images[2], method, output_dir / f"{stem}_02_03", ransac_iterations, min_inliers, min_inlier_ratio
    )
    h21 = np.linalg.inv(h12)
    h32 = np.linalg.inv(h23)

    transforms_by_reference = {
        "ref_1": [np.eye(3), h21, h21 @ h32],
        "ref_2": [h12, np.eye(3), h32],
        "ref_3": [h23 @ h12, h23, np.eye(3)],
    }

    h12_t = None
    h23_t = None

    for ref_name, transforms in transforms_by_reference.items():
        try:
            panorama = make_panorama(images, transforms)
        except RuntimeError as exc:
            print(f"retry translacao: {stem}_{ref_name} -> {exc}")
            if h12_t is None or h23_t is None:
                h12_t = estimate_pair_translation(images[0], images[1], method)
                h23_t = estimate_pair_translation(images[1], images[2], method)
            h21_t = np.linalg.inv(h12_t)
            h32_t = np.linalg.inv(h23_t)
            translation_transforms = {
                "ref_1": [np.eye(3), h21_t, h21_t @ h32_t],
                "ref_2": [h12_t, np.eye(3), h32_t],
                "ref_3": [h23_t @ h12_t, h23_t, np.eye(3)],
            }
            panorama = make_panorama(images, translation_transforms[ref_name], max_side=30000, max_area=250_000_000)
        out_path = output_dir / f"{stem}_{ref_name}.png"
        cv2.imwrite(str(out_path), panorama)
        print(f"salvo: {out_path}")

    note12 = " (fallback translacao)" if fallback12 else ""
    note23 = " (fallback translacao)" if fallback23 else ""
    print(f"{paths[0].name}-{paths[1].name}: {matches12} matches, {inliers12} inliers{note12}")
    print(f"{paths[1].name}-{paths[2].name}: {matches23} matches, {inliers23} inliers{note23}")


def find_trios(pattern):
    files = sorted(Path().glob(pattern))
    groups = {}
    for path in files:
        parts = path.stem.rsplit("_", 1)
        if len(parts) != 2:
            continue
        prefix, index = parts
        groups.setdefault(prefix, {})[index] = path

    trios = []
    for prefix, items in sorted(groups.items()):
        if {"01", "02", "03"}.issubset(items):
            trios.append([items["01"], items["02"], items["03"]])
    return trios


# Execução da Questão 1: processa automaticamente os trios 01..05 se estiverem na pasta.
output_dir = Path("q1_output")
trios = find_trios("q1_images/*_0[1-3].jpg")

if not trios:
    print("Nenhum trio encontrado. Use nomes como 01_01.jpg, 01_02.jpg, 01_03.jpg.")
else:
    for trio in trios:
        print("\ntrio:", ", ".join(str(path) for path in trio))
        process_trio(
            trio,
            method="SIFT",
            output_dir=output_dir,
            ransac_iterations=250,
            min_inliers=100,
            min_inlier_ratio=0.2,
        )

    # Exibe os panoramas gerados.
    panorama_paths = sorted(output_dir.glob("*_ref_*.png"))
    for path in panorama_paths:
        img = cv2.imread(str(path))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(16, 6))
        plt.imshow(img_rgb)
        plt.title(str(path))
        plt.axis("off")
        plt.show()


# Questão 2:

Considere a imagem `soccer.jpg`. Considere que o campo da imagem tenha dimensões 105 m x 68 m.

Gere manualmente correspondências entre a imagem e um mapa 2D com as dimensões dadas. Considere que o canto inferior esquerdo tem coordenada `(0,0)`. Calcule a homografia resultante e aplique na imagem original. Exiba o resultado. Informe as coordenadas resultantes dos 2 jogadores de vermelho próximos a um dos cantos do campo.


In [ ]:
FIELD_LENGTH_M = 105.0
FIELD_WIDTH_M = 68.0
OUTPUT_SCALE = 20

IMAGE_POINTS = np.float32(
    [
        [2888, 4380],  # near lower-left field corner
        [7161, 3490],  # near lower-right field corner
        [4156, 1251],  # far upper-right field corner
        [860, 1650],   # far upper-left field corner
    ]
)

FIELD_POINTS_M = np.float32(
    [
        [FIELD_LENGTH_M, 0.0],
        [FIELD_LENGTH_M, FIELD_WIDTH_M],
        [0.0, FIELD_WIDTH_M],
        [0.0, 0.0],
    ]
)

RED_PLAYERS_IMAGE = np.float32(
    [
        [3694, 1412],
        [3917, 1321],
    ]
).reshape(-1, 1, 2)

CENTER_FIELD_M = np.float32([[[FIELD_LENGTH_M / 2.0, FIELD_WIDTH_M / 2.0]]])


def field_to_output_pixels(points_m):
    """Convert field coordinates in meters to top-down output image pixels."""
    points_m = np.asarray(points_m, dtype=np.float32)
    x_px = points_m[:, 0] * OUTPUT_SCALE
    y_px = (FIELD_WIDTH_M - points_m[:, 1]) * OUTPUT_SCALE
    return np.column_stack([x_px, y_px]).astype(np.float32)


def draw_points(image, points, labels, color):
    out = image.copy()
    for point, label in zip(points.reshape(-1, 2), labels):
        x, y = np.round(point).astype(int)
        cv2.circle(out, (x, y), 14, color, -1)
        cv2.putText(
            out,
            label,
            (x + 18, y - 18),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            color,
            3,
            cv2.LINE_AA,
        )
    return out


def show_with_matplotlib(title, image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(image_rgb)
    ax.set_title(title)
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")
    Cursor(ax, useblit=True, color="yellow", linewidth=1)

    height, width = image_rgb.shape[:2]

    def format_coord(x, y):
        col = int(round(x))
        row = int(round(y))
        if 0 <= col < width and 0 <= row < height:
            r, g, b = image_rgb[row, col]
            return f"x={x:.1f}, y={y:.1f}, RGB=({r},{g},{b})"
        return f"x={x:.1f}, y={y:.1f}"

    def on_click(event):
        if event.inaxes == ax and event.xdata is not None and event.ydata is not None:
            print(f"{title}: click x={event.xdata:.1f}, y={event.ydata:.1f}")

    ax.format_coord = format_coord
    fig.canvas.mpl_connect("button_press_event", on_click)
    return fig



# Execução da Questão 2.
image_path = Path("q2_images/soccer.jpg")
image = cv2.imread(str(image_path))
if image is None:
    raise FileNotFoundError(f"Não foi possível ler a imagem: {image_path}")

h_img_to_field_m, _ = cv2.findHomography(IMAGE_POINTS, FIELD_POINTS_M)
h_field_m_to_img = np.linalg.inv(h_img_to_field_m)

output_size = (int(FIELD_LENGTH_M * OUTPUT_SCALE), int(FIELD_WIDTH_M * OUTPUT_SCALE))
output_points_px = field_to_output_pixels(FIELD_POINTS_M)
h_img_to_output_px, _ = cv2.findHomography(IMAGE_POINTS, output_points_px)
warped = cv2.warpPerspective(image, h_img_to_output_px, output_size)

players_field_m = cv2.perspectiveTransform(RED_PLAYERS_IMAGE, h_img_to_field_m)
players_output_px = field_to_output_pixels(players_field_m.reshape(-1, 2)).reshape(-1, 1, 2)
center_image = cv2.perspectiveTransform(CENTER_FIELD_M, h_field_m_to_img)
center_output_px = field_to_output_pixels(CENTER_FIELD_M.reshape(-1, 2)).reshape(-1, 1, 2)

annotated = draw_points(
    image,
    np.vstack([IMAGE_POINTS.reshape(-1, 1, 2), RED_PLAYERS_IMAGE]),
    ["(105,0)", "(105,68)", "(0,68)", "(0,0)", "P1", "P2"],
    (0, 0, 255),
)
annotated = draw_points(annotated, center_image, ["C=(52.5,34)"], (255, 0, 0))
warped_annotated = draw_points(
    warped,
    np.vstack([players_output_px, center_output_px]),
    ["P1", "P2", "C=(52.5,34)"],
    (0, 0, 255),
)
warped_annotated = draw_points(warped_annotated, center_output_px, ["C=(52.5,34)"], (255, 0, 0))

print("Homografia imagem -> campo (metros):")
print(h_img_to_field_m)
print("\nCoordenadas dos jogadores vermelhos no campo (metros):")
for i, (x, y) in enumerate(players_field_m.reshape(-1, 2), start=1):
    print(f"P{i}: x={x:.2f} m, y={y:.2f} m")
cx, cy = CENTER_FIELD_M.reshape(-1, 2)[0]
cix, ciy = center_image.reshape(-1, 2)[0]
print(f"\nCentro de teste: x={cx:.2f} m, y={cy:.2f} m")
print(f"Centro projetado na imagem original: px={cix:.1f}, py={ciy:.1f}")

Path("q2_output").mkdir(parents=True, exist_ok=True)
cv2.imwrite("q2_output/q2_soccer_original_anotada.png", annotated)
cv2.imwrite("q2_output/q2_soccer_warped.png", warped_annotated)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("Original anotada")
plt.axis("off")
plt.show()

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(warped_annotated, cv2.COLOR_BGR2RGB))
plt.title("Campo retificado")
plt.axis("off")
plt.show()


# Questão 3:

Leia o tutorial de calibração de câmera da OpenCV. Use um tabuleiro de xadrez medido, considerando o plano `z = 0` e a origem no canto inferior esquerdo. Após calibrar a câmera, inclua um objeto virtual na imagem.

Considere o círculo paramétrico centrado em `(1.5W, 1.5H, 0)`, com raio `r = 0.5W`, contido no plano `z = 0`. Varie `theta` entre `0` e `2π`, projete os pontos na imagem e repita o experimento 3 vezes variando o ângulo entre o vetor normal do tabuleiro e o eixo principal da câmera.


# Questão 4:

Simule o efeito de fotografia em modo retrato usando mapas de disparidade gerados a partir de duas imagens.

A partir do mapa de disparidade, detecte o foreground, aplique filtro Gaussiano apenas no background e combine as partes para obter o efeito de fundo borrado. Experimente em três pares de imagens capturadas com pequenas variações de ponto de vista da câmera.
